# 02 - Data Preparation

Cleans and merges all raw datasets into a single analysis-ready CSV with one row per county.

In [1]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
from src.clean import (
    compute_two_party_share,
    compute_swing,
    process_acs,
    parse_bls_laus,
    parse_nchs,
    parse_gazetteer,
)

RAW = os.path.join('..', 'data', 'raw')
PROCESSED = os.path.join('..', 'data', 'processed')

## Election data: two-party vote share and swing

In [2]:
election = pd.read_csv(os.path.join(RAW, 'countypres_2000-2024.csv'), dtype=str)

# MEDSL may use 'county_fips' or 'FIPS' depending on version
fips_candidates = [c for c in election.columns if 'fips' in c.lower()]
print(f"FIPS column candidates: {fips_candidates}")
fips_col = fips_candidates[0]
election = election.rename(columns={fips_col: 'county_fips'})

# convert numeric columns
election['year'] = pd.to_numeric(election['year'], errors='coerce')
election['candidatevotes'] = pd.to_numeric(election['candidatevotes'], errors='coerce')
election['totalvotes'] = pd.to_numeric(election['totalvotes'], errors='coerce')

# drop rows with missing FIPS (write-in aggregates, header artifacts)
election = election.dropna(subset=['county_fips'])
election = election[election['county_fips'].str.match(r'^\d+$')]
election['county_fips'] = election['county_fips'].str.zfill(5)

print(f"Election data: {len(election)} rows")
print(f"Years available: {sorted(election['year'].unique())}")
print(f"Unique counties: {election['county_fips'].nunique()}")

FIPS column candidates: ['county_fips']
Election data: 94099 rows
Years available: [np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020), np.int64(2024)]
Unique counties: 3157


In [3]:
shares = compute_two_party_share(election)
swing = compute_swing(shares)

print(f"Counties with swing data: {len(swing)}")
print(f"\nSwing summary (positive = more Republican):")
print(f"  Mean:   {swing['swing'].mean():.4f} ({swing['swing'].mean()*100:.2f} pp)")
print(f"  Median: {swing['swing'].median():.4f} ({swing['swing'].median()*100:.2f} pp)")
print(f"  Std:    {swing['swing'].std():.4f}")
print(f"  Min:    {swing['swing'].min():.4f}")
print(f"  Max:    {swing['swing'].max():.4f}")

Counties with swing data: 3152

Swing summary (positive = more Republican):
  Mean:   0.0172 (1.72 pp)
  Median: 0.0155 (1.55 pp)
  Std:    0.0251
  Min:    -0.3384
  Max:    0.4758


## ACS demographics

In [4]:
acs = pd.read_csv(os.path.join(RAW, 'acs_demographics.csv'), dtype={'FIPS': str})
acs['FIPS'] = acs['FIPS'].str.zfill(5)
acs_clean = process_acs(acs)

print(f"ACS counties: {len(acs_clean)}")
acs_clean.describe()

ACS counties: 3222


,total_pop,median_age,median_hh_income,pct_college,pct_unemployed,pct_white_nh,pct_black,pct_hispanic
count,3.222000e+03,3222.00000,3220.000000,3222.000000,3222.000000,3222.000000,3222.000000,3222.000000
mean,1.041721e+05,41.82486,65046.649068,0.240993,0.049205,0.725179,0.087343,0.122146
std,3.296689e+05,5.36581,18388.683350,0.101127,0.027770,0.228270,0.140441,0.193166
min,4.300000e+01,21.40000,16170.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.098550e+04,38.80000,54113.250000,0.170794,0.033303,0.608907,0.007243,0.025716
50%,2.596700e+04,41.50000,63161.500000,0.215273,0.045266,0.804475,0.022739,0.050648
75%,6.752625e+04,44.60000,73216.250000,0.286757,0.058972,0.904832,0.097250,0.117420
max,9.848406e+06,68.90000,178707.000000,0.797257,0.311024,1.000000,0.862259,1.000000


## NCHS urban-rural classification

In [5]:
nchs = parse_nchs(os.path.join(RAW, 'nchs_urban_rural.csv'))
print(f"NCHS counties: {len(nchs)}")
nchs['urban_rural_label'].value_counts()

NCHS counties: 3160


urban_rural_label
Noncore (rural)        1335
Micropolitan            641
Medium metro            373
Large fringe metro      368
Small metro             358
Large central metro      68
Name: count, dtype: int64

## Gazetteer (land area)

In [6]:
gaz = parse_gazetteer(os.path.join(RAW, 'gazetteer_counties.txt'))
print(f"Gazetteer counties: {len(gaz)}")
gaz.head()

Gazetteer counties: 3222


,FIPS,land_area_sqmi
0,01001,594.455
1,01003,1589.884
2,01005,885.008
3,01007,622.470
4,01009,644.891


## BLS state unemployment

In [7]:
bls = parse_bls_laus(os.path.join(RAW, 'bls_laus_states.json'))
print(f"States with unemployment data: {len(bls)}")
bls.head()

States with unemployment data: 51


year,state_fips,unemp_rate_2020,unemp_rate_2024,unemp_rate_change
0,01,6.408333,3.058333,-3.350000
1,02,8.283333,4.591667,-3.691667
2,04,7.825000,3.600000,-4.225000
3,05,6.158333,3.466667,-2.691667
4,06,10.166667,5.316667,-4.850000


## Merge all datasets

In [8]:
# start from the swing table (unit of analysis = county)
df = swing.rename(columns={'county_fips': 'FIPS'}).copy()
n_start = len(df)

# join ACS demographics
df = df.merge(acs_clean, on='FIPS', how='left')
print(f"After ACS join: {len(df)} rows (started with {n_start})")

# join NCHS urban-rural
df = df.merge(nchs, on='FIPS', how='left')
print(f"After NCHS join: {len(df)} rows")

# join gazetteer for land area
df = df.merge(gaz, on='FIPS', how='left')
print(f"After gazetteer join: {len(df)} rows")

# compute population density
df['pop_density'] = df['total_pop'] / df['land_area_sqmi']

# join BLS state unemployment change
df['state_fips'] = df['FIPS'].str[:2]
df = df.merge(bls, on='state_fips', how='left')
print(f"After BLS join: {len(df)} rows")

After ACS join: 3152 rows (started with 3152)
After NCHS join: 3152 rows
After gazetteer join: 3152 rows
After BLS join: 3152 rows


In [9]:
# flag Alaska (different reporting structure)
df['is_alaska'] = df['FIPS'].str.startswith('02')
print(f"Alaska rows: {df['is_alaska'].sum()}")

# log-transform skewed variables
df['log_pop_density'] = np.log1p(df['pop_density'])
df['log_median_income'] = np.log1p(df['median_hh_income'])

Alaska rows: 41


## Quality checks

In [10]:
print(f"Total counties: {len(df)}")
print(f"Duplicated FIPS: {df['FIPS'].duplicated().sum()}")
print(f"\nMissing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

print(f"\nComplete cases (no NaN in key columns): ", end='')
key_cols = ['swing', 'pct_college', 'pct_hispanic', 'pct_white_nh',
            'median_hh_income', 'pct_unemployed', 'pop_density', 'median_age']
complete = df.dropna(subset=key_cols)
print(f"{len(complete)} / {len(df)} ({100*len(complete)/len(df):.1f}%)")

Total counties: 3152
Duplicated FIPS: 0

Missing values per column:
NAME                 46
total_pop            46
median_age           46
median_hh_income     48
pct_college          46
pct_unemployed       46
pct_white_nh         46
pct_black            46
pct_hispanic         46
urban_rural_code     38
urban_rural_label    38
land_area_sqmi       46
pop_density          46
log_pop_density      46
log_median_income    48

Complete cases (no NaN in key columns): 3104 / 3152 (98.5%)


In [11]:
# counties per state
state_counts = df.groupby('state_fips').size().reset_index(name='n_counties')
print(f"States represented: {len(state_counts)}")
print(f"Counties per state: min={state_counts['n_counties'].min()}, "
      f"max={state_counts['n_counties'].max()}, "
      f"median={state_counts['n_counties'].median():.0f}")

States represented: 51
Counties per state: min=1, max=254, median=62


In [12]:
# check for extreme outliers in swing
extreme = df[df['swing'].abs() > 0.3]  # > 30 percentage points
if len(extreme) > 0:
    print(f"Extreme swing outliers (>30pp): {len(extreme)}")
    print(extreme[['FIPS', 'NAME', 'swing', 'total_pop']].to_string())
else:
    print("No extreme outliers (>30pp swing)")

Extreme swing outliers (>30pp): 3
     FIPS NAME     swing  total_pop
69  02003  NaN -0.338386        NaN
93  02027  NaN  0.318048        NaN
99  02033  NaN  0.475848        NaN


In [13]:
# save
os.makedirs(PROCESSED, exist_ok=True)
out_path = os.path.join(PROCESSED, 'county_swing_dataset.csv')
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"{len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

Saved: ..\data\processed\county_swing_dataset.csv
3152 rows, 28 columns
Columns: ['FIPS', 'rep_share_2020', 'dem_votes_2020', 'rep_votes_2020', 'rep_share_2024', 'dem_votes_2024', 'rep_votes_2024', 'swing', 'NAME', 'total_pop', 'median_age', 'median_hh_income', 'pct_college', 'pct_unemployed', 'pct_white_nh', 'pct_black', 'pct_hispanic', 'urban_rural_code', 'urban_rural_label', 'land_area_sqmi', 'pop_density', 'state_fips', 'unemp_rate_2020', 'unemp_rate_2024', 'unemp_rate_change', 'is_alaska', 'log_pop_density', 'log_median_income']
